# Migration Phase 7: multivariate regional BAG via `regional-stacker`

Phase 7 adds `MultivariateRegionalBAGEstimator`
(`src/neuroalign/modeling/multivariate/estimator.py`), which combines
multiple wide-format feature tables (e.g. cortical thickness + diffusion FA)
into a single per-region feature block via
`regional_stacker.wide_to_stacker_input`, then fits a
`RegionalStackingRegressor` (per-region base learners + a meta-learner) under
subject-grouped cross-validation to predict chronological age.

As with the univariate estimator
(`src/neuroalign/modeling/univariate/estimator.py`), `BAG = predicted_age -
actual_age`, with optional De Lange & Cole post-hoc bias correction. The
shared IPW/bias-correction helpers now live in `src/neuroalign/modeling/_shared.py`
and are reused by both estimators.

This notebook runs the new estimator end-to-end against the Phase 5 demo
store, combining one anatomical and one diffusion feature.

In [1]:
from pathlib import Path
from dotenv import load_dotenv

load_dotenv(Path.cwd().parent / ".env")

from neuroalign.data.preprocessing import FeatureStore
from neuroalign.modeling import BAGConfig, MultivariateRegionalBAGEstimator

store_dir = Path.cwd().parent / "data" / "processed" / "phase5_demo"
store = FeatureStore(store_dir)

meta = store.load_metadata()
print(f"n_sessions={len(meta)}, n_subjects={meta['uid'].nunique()}")
meta.head()

n_sessions=46, n_subjects=42


,session_id,uid,subject_code,subject_code_bids,lab,scan_date,scan_tag,scan_number,protocol,study,group_label,mapping_complete,AGE,sex,weight_kg,height_m,dominant_hand
0,202410131245,S076379,BB01028,None,TS,2024-10-13,None,2,Brain Bank,TS,None,True,22.26,Female,67.0,1.61,None
1,202410071918,S606615,BB01023,None,TS,2024-10-07,None,1,Brain Bank,TS,None,True,30.74,Male,85.0,1.74,None
2,202410101257,S656985,BB01025,None,TS,2024-10-10,None,1,Brain Bank,TS,None,True,28.80,Female,59.0,1.58,None
3,202510160926,S108449,BB01018,None,TS,2025-10-16,None,2,Brain Bank,TS,None,True,31.28,Female,80.0,1.58,None
4,202507241349,S028694,BB01012,None,TS,2025-07-24,None,3,Brain Bank,TS,None,True,23.81,Female,67.0,1.64,None


## 1. Combine an anatomical and a diffusion feature

`feature_names` lists the wide-format tables to combine. Each table is
reindexed to the (uid, session_id) intersection across all tables and
concatenated per region.

In [2]:
feature_names = ["anat_thickness_mean_mm", "DSIStudio_tensor_fa_mean"]

for name in feature_names:
    df = store.load_feature(name, include_metadata=False)
    print(f"{name}: {df.shape[0]} sessions x {df.shape[1] - 2} regions")

anat_thickness_mean_mm: 12 sessions x 400 regions
DSIStudio_tensor_fa_mean: 22 sessions x 432 regions


## 2. Fit the multivariate estimator

`BAGConfig.age_col` defaults to `"age"`, but `FeatureStore.load_metadata()`
exposes the age column as `"AGE"` - override it accordingly. With 19
subjects, the default `n_splits=5` GroupKFold is fine.

In [3]:
config = BAGConfig(age_col="AGE")
estimator = MultivariateRegionalBAGEstimator(config)

result = estimator.fit_predict(store, feature_names)

/home/galkepler/Projects/neuroalign/src/neuroalign/modeling/multivariate/estimator.py:165: UserWarning: 10 subject(s) absent from one or more tables and excluded from the intersection: [('S019804', '202410071148'), ('S166352', '202410080846'), ('S203749', '202411031340'), ('S483889', '202409301535'), ('S593447', '202410310932'), ('S606615', '202410071918'), ('S656985', '202410101257'), ('S736879', '202410301546'), ('S787835', '202411070913'), ('S829190', '202410141135')]
  x, region_mapping, sessions = wide_to_stacker_input(tables)


Outer folds:   0%|          | 0/5 [00:00<?, ?it/s]

## 3. Inspect results

`bag`/`bag_uncorrected`/`predicted_age` are session-level (one row per
session, single value column) since the multivariate model produces one age
prediction per session. `region_metrics` reports per-region stage-1 (base
learner) out-of-fold diagnostics from a full-data fit.

In [4]:
result.predicted_age.head()

,uid,session_id,predicted_age
0,S028694,202409261650,28.151723
1,S028694,202507241349,28.171898
2,S067082,202509031744,29.506718
3,S076379,202410131245,30.828233
4,S108449,202510160926,31.991554


In [5]:
result.bag.describe()

,bag
count,1.200000e+01
mean,-7.993606e-15
std,1.311100e+00
min,-1.459974e+00
25%,-1.108034e+00
50%,-1.629357e-01
75%,6.265717e-01
max,2.349811e+00


In [6]:
result.region_metrics.sort_values("r2", ascending=False).head(10)

,region,n_features,r2,mae,correlation
421,LH_Cont_pCun_2,1,0.798193,1.640383,0.894566
542,LH_SomMot_19,1,0.570186,2.433745,0.780489
203,7Networks_RH_Cont_PFCl_10,1,0.566429,2.828602,0.757093
511,LH_SalVentAttn_FrOperIns_2,1,0.559451,2.632594,0.770856
82,7Networks_LH_DorsAttn_Post_13,1,0.555905,2.688131,0.749078
272,7Networks_RH_DorsAttn_Post_1,1,0.515611,2.877807,0.736365
527,LH_SalVentAttn_ParOper_1,1,0.513669,2.869362,0.748499
574,LH_Vis_14,1,0.452319,3.248562,0.700238
770,RH_SomMot_6,1,0.433087,3.222092,0.729697
25,7Networks_LH_Default_PFC_12,1,0.432288,2.807175,0.692858


## 4. Overall predictive performance

Compare predicted vs actual age across the held-out folds.

In [7]:
from sklearn.metrics import mean_absolute_error, r2_score

merged = result.predicted_age.merge(meta[["uid", "session_id", "AGE"]], on=["uid", "session_id"])
mae = mean_absolute_error(merged["AGE"], merged["predicted_age"])
r2 = r2_score(merged["AGE"], merged["predicted_age"])
print(f"MAE={mae:.2f} years, R2={r2:.3f}")
merged[["uid", "session_id", "AGE", "predicted_age"]].head()

MAE=3.84 years, R2=0.017


,uid,session_id,AGE,predicted_age
0,S028694,202409261650,22.99,28.151723
1,S028694,202507241349,23.81,28.171898
2,S067082,202509031744,37.49,29.506718
3,S076379,202410131245,22.26,30.828233
4,S108449,202510160926,31.28,31.991554
